#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.fact_sales"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension fact_sales

###Fetch Relative Columns

In [0]:
# Join sales with product and customer dimensions
df_scr = spark.sql('''
    SELECT
        --ROW_NUMBER() OVER (ORDER BY sd.order_date,sd.order_number) AS fact_sale_surrogate_key, 
        sd.order_number,
        pr.product_surrogate_key,
        cu.customer_surrogate_key,
        sd.order_date,
        sd.ship_date,
        sd.due_date,
        sd.sales_amount,
        sd.quantity,
        sd.price
    FROM store_catalog.gold.dim_crm_sales_details AS sd
    LEFT JOIN store_catalog.gold.dim_products AS pr
        ON sd.product_number = pr.product_number
    LEFT JOIN store_catalog.gold.dim_customers AS cu
        ON sd.customer_id = cu.customer_id;
''')

In [0]:
df_scr.limit(10).display()

order_number,product_surrogate_key,customer_surrogate_key,order_date,ship_date,due_date,sales_amount,quantity,price
SO65608,266,12227,2013-09-02,2013-09-09,2013-09-14,1215.0,1,1215.0
SO52027,361,15127,2013-01-31,2013-02-07,2013-02-12,35.0,1,35.0
SO52027,240,15127,2013-01-31,2013-02-07,2013-02-12,35.0,1,35.0
SO52027,79,15127,2013-01-31,2013-02-07,2013-02-12,35.0,1,35.0
SO57411,386,4533,2013-05-03,2013-05-10,2013-05-15,10.0,1,10.0
SO66241,189,4899,2013-09-12,2013-09-19,2013-09-24,35.0,1,35.0
SO66241,67,4899,2013-09-12,2013-09-19,2013-09-24,35.0,1,35.0
SO66241,18,4899,2013-09-12,2013-09-19,2013-09-24,35.0,1,35.0
SO48958,269,14493,2012-07-20,2012-07-27,2012-08-01,2182.0,1,2182.0
SO48958,141,14493,2012-07-20,2012-07-27,2012-08-01,2182.0,1,2182.0


###fact_sales - Initial and Incremental

In [0]:
# Read existing fact records or create an empty DataFrame
if spark.catalog.tableExists("store_catalog.gold.fact_sales") == True:
    df_sink = spark.sql('''
            SELECT
                fact_sale_surrogate_key,
                order_number,
                product_surrogate_key,
                customer_surrogate_key,
                order_date,
                ship_date,
                due_date,
                sales_amount,
                quantity,
                price
            FROM
                store_catalog.gold.fact_sales
        ''')
else:
    df_sink = spark.sql('''
            SELECT
                1 as fact_sale_surrogate_key, 
                sd.order_number,
                pr.product_surrogate_key,
                cu.customer_surrogate_key,
                sd.order_date,
                sd.ship_date,
                sd.due_date,
                sd.sales_amount,
                sd.quantity,
                sd.price
            FROM store_catalog.gold.dim_crm_sales_details AS sd
            LEFT JOIN store_catalog.gold.dim_products AS pr
                ON sd.product_number = pr.product_number
            LEFT JOIN store_catalog.gold.dim_customers AS cu
                ON sd.customer_id = cu.customer_id
            WHERE
                1 = 0
        ''')

In [0]:
# Check for duplicate business key combinations
df_sink.groupBy("order_number","product_surrogate_key","customer_surrogate_key") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+------------+---------------------+----------------------+-----+
|order_number|product_surrogate_key|customer_surrogate_key|count|
+------------+---------------------+----------------------+-----+
+------------+---------------------+----------------------+-----+



###Filtering new records and old records

In [0]:
# Match sales using order, product, and customer keys
df_filter = df_scr.join(df_sink,(df_scr["order_number"] == df_sink["order_number"]) & (df_scr["product_surrogate_key"] == df_sink["product_surrogate_key"]) & (df_scr["customer_surrogate_key"] == df_sink["customer_surrogate_key"]), "left")\
        .select(df_scr["order_number"],df_scr["product_surrogate_key"],df_scr["customer_surrogate_key"],
        df_scr["order_date"],df_scr["ship_date"],df_scr["due_date"],df_scr["sales_amount"],df_scr["quantity"],df_scr["price"],df_sink["fact_sale_surrogate_key"]
    )

In [0]:
# Validate business key uniqueness
df_filter.groupBy("order_number","product_surrogate_key","customer_surrogate_key") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+------------+---------------------+----------------------+-----+
|order_number|product_surrogate_key|customer_surrogate_key|count|
+------------+---------------------+----------------------+-----+
+------------+---------------------+----------------------+-----+



**df_filter_old**

In [0]:
# Separate existing fact records
df_filter_old = df_filter.filter(col("fact_sale_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Identify new sales records requiring surrogate keys
df_filter_new = df_filter.filter(col("fact_sale_surrogate_key").isNull())\
    .select(df_scr["order_number"],df_scr["product_surrogate_key"],df_scr["customer_surrogate_key"],
            df_scr["order_date"],df_scr["ship_date"],df_scr["due_date"],df_scr["sales_amount"],
            df_scr["quantity"],df_scr["price"]
    )

In [0]:
df_filter_new.display()

order_number,product_surrogate_key,customer_surrogate_key,order_date,ship_date,due_date,sales_amount,quantity,price


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the current maximum surrogate key
if incremental_flag == "0":
    max_value = 0
else:
    max_value = spark.sql("""
        SELECT COALESCE(MAX(fact_sale_surrogate_key), 0)
        FROM store_catalog.gold.fact_sales
    """).collect()[0][0]

In [0]:
print("incremental_flag:", incremental_flag)
print("max_value:", max_value)

incremental_flag: 1
max_value: 89818


**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign sequential surrogate keys to new fact records
window_spec = Window.orderBy(
    "order_number",
    "product_surrogate_key",
    "customer_surrogate_key"
)

df_filter_new = df_filter_new.withColumn(
    "fact_sale_surrogate_key",
    (row_number().over(window_spec) + max_value).cast("long")
)

###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine existing sales records with newly identified records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.display()

order_number,product_surrogate_key,customer_surrogate_key,order_date,ship_date,due_date,sales_amount,quantity,price,fact_sale_surrogate_key
SO72655,384,14455,2013-12-09,2013-12-16,2013-12-21,2320.0,1,2320.0,81050
SO72655,139,14455,2013-12-09,2013-12-16,2013-12-21,2320.0,1,2320.0,81046
SO55075,31,3354,2013-03-26,2013-04-02,2013-04-07,1701.0,1,1701.0,21434
SO61993,351,18235,2013-07-12,2013-07-19,2013-07-24,22.0,1,22.0,44673
SO72799,52,3834,2013-12-11,2013-12-18,2013-12-23,4.0,1,4.0,81606
SO73168,393,10856,2013-12-15,2013-12-22,2013-12-27,9.0,1,9.0,82969
SO73168,329,10856,2013-12-15,2013-12-22,2013-12-27,9.0,1,9.0,82967
SO73168,182,10856,2013-12-15,2013-12-22,2013-12-27,9.0,1,9.0,82965
SO69041,357,5346,2013-10-24,2013-10-31,2013-11-05,2.0,1,2.0,68811
SO45173,75,2298,2011-07-17,2011-07-24,2011-07-29,3578.0,1,3578.0,1231


#SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.fact_sales"):
    # Incremental Run
    # Merge fact sales records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/fact_sales")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.fact_sale_surrogate_key = src.fact_sale_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/fact_sales")\
        .saveAsTable("store_catalog.gold.fact_sales")

In [0]:
%sql
SELECT * FROM store_catalog.gold.fact_sales  ORDER BY fact_sale_surrogate_key LIMIT 10

order_number,product_surrogate_key,customer_surrogate_key,order_date,ship_date,due_date,sales_amount,quantity,price,fact_sale_surrogate_key
SO43697,181,10508,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0,1
SO43698,143,5082,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,2
SO43699,143,7034,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,3
SO43700,117,16171,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0,4
SO43700,125,16171,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0,5
SO43701,143,8796,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,6
SO43702,170,12623,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0,7
SO43703,181,13975,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0,8
SO43704,299,293,2010-12-30,2011-01-06,2011-01-11,3375.0,1,3375.0,9
SO43705,322,7913,2010-12-30,2011-01-06,2011-01-11,3400.0,1,3400.0,10


In [0]:
%sql
SELECT * FROM store_catalog.gold.fact_sales  ORDER BY fact_sale_surrogate_key DESC LIMIT 10

order_number,product_surrogate_key,customer_surrogate_key,order_date,ship_date,due_date,sales_amount,quantity,price,fact_sale_surrogate_key
SO75123,393,4286,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89818
SO75123,351,4286,2014-01-28,2014-02-04,2014-02-09,22.0,1,22.0,89817
SO75123,329,4286,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89816
SO75123,182,4286,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89815
SO75123,118,4286,2014-01-28,2014-02-04,2014-02-09,159.0,1,159.0,89814
SO75122,393,6794,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89813
SO75122,351,6794,2014-01-28,2014-02-04,2014-02-09,22.0,1,22.0,89812
SO75122,329,6794,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89811
SO75122,182,6794,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0,89810
SO75121,361,1663,2014-01-28,2014-02-04,2014-02-09,35.0,1,35.0,89809


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.fact_sales 

count(1)
89818
